In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .appName('Good')\
            .master('yarn')\
                .getOrCreate()

In [0]:
schema = "id string , firstname string ,lastname string , address string , skills string , contacts string"

df = spark.read.format('csv')\
                .option('header' , 'true')\
                    .option('quote' , "\"")\
                        .option('escape' , "\"")\
                    .schema(schema)\
                        .load(path = '/Volumes/dev/spark_db/datasets/spark_programming/data/students_offline.csv')

In [0]:
df.display()

In [0]:
df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students_raw")

In [0]:
spark.sql('''select * from dev.spark_db.offline_students_raw''').display()

In [0]:
%sql

select * from dev.spark_db.offline_students_raw;

In [0]:
%sql 
       with offline_student as 
        (select id , from_json(address , 
                        '''AddressLine1 string,
                           AddressLine2 string ,
                           City string ,
                           Country string ,
                           Pin string ,
                           State string ''') as address
        from dev.spark_db.offline_students_raw )
      select address.country , count(*) as count from offline_student
      group by address.country ;       


In [0]:
spark.sql(''' with offline_student as 
        (select id , from_json(address , 
                        "AddressLine1 string,
                           AddressLine2 string ,
                           City string ,
                           Country string ,
                           Pin string ,
                           State string ") as address
        from dev.spark_db.offline_students_raw )
      select address.country , count(*) as count from offline_student
      group by address.country ;    
          
          ''').show()

In [0]:
spark.sql('''
          select * from dev.spark_db.offline_students_raw
          
          ''').display()

In [0]:
from pyspark.sql.functions import from_json

address_schema = "struct< AddressLine1 string, AddressLine2 string, City string ,Country string ,Pin string ,State string>"
skills_schema = "array<struct<Skill string , Yearsofexperiance int>>"
contacts_schema = "map<string , string>"  #no normal format

df = df.withColumns(
    {
        "address" : from_json("address" , address_schema),
        "skills" : from_json("skills" , skills_schema),
        "contacts" : from_json("contacts" ,  contacts_schema )
    }
)

df.write.mode('overwrite').saveAsTable("dev.spark_db.offline_students")

print("Successfully Inserted")

In [0]:
spark.sql("""
          
          select address['country'] , count(*)from dev.spark_db.offline_students
          group by  address['country']

          """).display()

In [0]:
spark.sql("""
          

          select id , firstname , lastname , skill['Skill'] ,  skill['Yearsofexperiance'] from (
              
          select id , firstname , lastname , explode(skills) as skill from dev.spark_db.offline_students
          
          ) d

          where skill['Skill'] = 'Apache Spark' 
          

          """).display()

In [0]:
spark.sql("""
          

          select id , firstname , lastname ,contacts['email']  from dev.spark_db.offline_students
          where contacts['phone'] is null and contacts['whatsapp'] is null
          

          """).display()